In [12]:
!pip -q install langchain langchain-community
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install pypdf
!pip -q install transformers
!pip -q install accelerate
!pip -q install langchain-text-splitters

In [13]:
import os
import numpy as np
import faiss

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from transformers import pipeline

In [14]:
uploaded = files.upload()

Saving 2G54QACZZK5MIIKK25USTLNPN66FST63.pdf to 2G54QACZZK5MIIKK25USTLNPN66FST63 (1).pdf


In [15]:
pdf_file = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_file)

Uploaded PDF: 2G54QACZZK5MIIKK25USTLNPN66FST63 (1).pdf


In [16]:
loader = PyPDFLoader(pdf_file)

documents = loader.load()

In [17]:
print("Total Pages:", len(documents))

Total Pages: 8


In [18]:
text = ""

for page in documents:
    text += page.page_content + "\n"

In [19]:
print(text[:2000])

Current Forecast: December 6, 2016; Previous Forecast: November 8, 2016
Q1 Q2 Q3 Q4 Q1 Q2 Q3 Q4 Q1 Q2 Q3 Q4 2014 2015 2016 2017 2014-2015 2015-2016 2016-2017
U.S. Energy Supply
   U.S. Crude Oil Production (million barrels per day)
      Current 9.49 9.47 9.41 9.30 9.17 8.85 8.67 8.75 8.75 8.74 8.70 8.94 8.76 9.42 8.86 8.78 7.4% -5.9% -0.9%
      Previous 9.49 9.47 9.41 9.30 9.17 8.85 8.68 8.68 8.68 8.71 8.67 8.87 8.76 9.42 8.84 8.73 7.4% -6.1% -1.3%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.1% 0.7% 0.7% 0.4% 0.3% 0.8% 0.0% 0.0% 0.2% 0.6%
   U.S. Dry Natural Gas Production (billion cubic feet per day)
      Current 73.44 74.50 74.51 74.08 73.77 72.38 71.89 71.94 73.22 74.48 75.34 76.13 70.93 74.14 72.49 74.80 4.5% -2.2% 3.2%
      Previous 73.44 74.50 74.51 74.08 73.77 72.38 71.73 71.51 73.06 74.41 75.81 76.93 70.93 74.14 72.34 75.06 4.5% -2.4% 3.8%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.2% 0.6% 0.2% 0.1% -0.6% -1.0% 0.0% 0.0% 0.2% -0.4%
   U.S. Coal Pro

In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(text)

In [21]:
print("Total Chunks:", len(chunks))

Total Chunks: 89


In [22]:
chunks[:5]

['Current Forecast: December 6, 2016; Previous Forecast: November 8, 2016\nQ1 Q2 Q3 Q4 Q1 Q2 Q3 Q4 Q1 Q2 Q3 Q4 2014 2015 2016 2017 2014-2015 2015-2016 2016-2017\nU.S. Energy Supply\n   U.S. Crude Oil Production (million barrels per day)\n      Current 9.49 9.47 9.41 9.30 9.17 8.85 8.67 8.75 8.75 8.74 8.70 8.94 8.76 9.42 8.86 8.78 7.4% -5.9% -0.9%\n      Previous 9.49 9.47 9.41 9.30 9.17 8.85 8.68 8.68 8.68 8.71 8.67 8.87 8.76 9.42 8.84 8.73 7.4% -6.1% -1.3%',
 'Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.1% 0.7% 0.7% 0.4% 0.3% 0.8% 0.0% 0.0% 0.2% 0.6%\n   U.S. Dry Natural Gas Production (billion cubic feet per day)\n      Current 73.44 74.50 74.51 74.08 73.77 72.38 71.89 71.94 73.22 74.48 75.34 76.13 70.93 74.14 72.49 74.80 4.5% -2.2% 3.2%\n      Previous 73.44 74.50 74.51 74.08 73.77 72.38 71.73 71.51 73.06 74.41 75.81 76.93 70.93 74.14 72.34 75.06 4.5% -2.4% 3.8%',
 'Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.2% 0.6% 0.2% 0.1% -0.6% -1.0% 0.0% 0.0% 0.2% -0.4%\n   U.S. Coal Pr

In [23]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i+1}\n")
    print(chunk)
    print("-" * 80)


Chunk 1

Current Forecast: December 6, 2016; Previous Forecast: November 8, 2016
Q1 Q2 Q3 Q4 Q1 Q2 Q3 Q4 Q1 Q2 Q3 Q4 2014 2015 2016 2017 2014-2015 2015-2016 2016-2017
U.S. Energy Supply
   U.S. Crude Oil Production (million barrels per day)
      Current 9.49 9.47 9.41 9.30 9.17 8.85 8.67 8.75 8.75 8.74 8.70 8.94 8.76 9.42 8.86 8.78 7.4% -5.9% -0.9%
      Previous 9.49 9.47 9.41 9.30 9.17 8.85 8.68 8.68 8.68 8.71 8.67 8.87 8.76 9.42 8.84 8.73 7.4% -6.1% -1.3%
--------------------------------------------------------------------------------

Chunk 2

Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.1% 0.7% 0.7% 0.4% 0.3% 0.8% 0.0% 0.0% 0.2% 0.6%
   U.S. Dry Natural Gas Production (billion cubic feet per day)
      Current 73.44 74.50 74.51 74.08 73.77 72.38 71.89 71.94 73.22 74.48 75.34 76.13 70.93 74.14 72.49 74.80 4.5% -2.2% 3.2%
      Previous 73.44 74.50 74.51 74.08 73.77 72.38 71.73 71.51 73.06 74.41 75.81 76.93 70.93 74.14 72.34 75.06 4.5% -2.4% 3.8%
-------------------------------

In [24]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

print("Embeddings Shape:", embeddings.shape)

Embeddings Shape: (89, 384)


In [26]:
dimension = embeddings.shape[1]

print("Embedding Dimension:", dimension)

Embedding Dimension: 384


In [27]:
index = faiss.IndexFlatL2(dimension)

In [28]:
index.add(embeddings)

print("Total Vectors Stored:", index.ntotal)

Total Vectors Stored: 89


In [29]:
query = "What is the main topic of this document?"

In [30]:
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

In [31]:
k = 3

distances, indices = index.search(
    query_embedding,
    k
)

In [32]:
print("Distances:")

print(distances)

Distances:
[[1.5975424 1.6104785 1.7119825]]


In [33]:
print("Retrieved Chunk Indices:")

print(indices)

Retrieved Chunk Indices:
[[28 27 24]]


In [34]:
retrieved_chunks = []

for idx in indices[0]:

    retrieved_chunks.append(
        chunks[idx]
    )

In [35]:
for i, chunk in enumerate(retrieved_chunks):

    print(f"\nRetrieved Chunk {i+1}\n")

    print(chunk)

    print("-"*80)


Retrieved Chunk 1

OECD (Organization for Economic Cooperation and Development) Consumption
      Current 46.63 45.64 46.92 46.46 46.72 45.97 46.64 47.13 47.02 45.95 46.90 47.58 45.86 46.41 46.62 46.87 1.2% 0.4% 0.5%
      Previous 46.63 45.64 46.92 46.46 46.72 45.98 46.52 47.15 46.97 45.91 46.84 47.44 45.86 46.41 46.59 46.79 1.2% 0.4% 0.4%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.3% 0.0% 0.1% 0.1% 0.1% 0.3% 0.0% 0.0% 0.1% 0.2%
   Non-OECD Consumption
--------------------------------------------------------------------------------

Retrieved Chunk 2

Previous 94.73 95.48 96.46 96.50 95.53 95.51 96.26 97.31 96.49 97.42 97.70 98.10 93.35 95.80 96.16 97.43 2.6% 0.4% 1.3%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.1% -0.1% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0%
World Crude Oil and Liquid Fuels Consumption (million barrels per day)
   OECD (Organization for Economic Cooperation and Development) Consumption
------------------------------------------------------

In [37]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

In [38]:
user_question = input("Enter your question: ")

Enter your question: 2


In [39]:
question_embedding = embedding_model.encode(
    [user_question],
    convert_to_numpy=True
)

In [40]:
k = 3

distances, indices = index.search(
    question_embedding,
    k
)

In [41]:
retrieved_context = ""

for idx in indices[0]:

    retrieved_context += chunks[idx]

    retrieved_context += "\n\n"

In [42]:
print("Retrieved Context:\n")

print(retrieved_context)

Retrieved Context:

U.S. Industrial
      Current 2,623 2,743 2,835 2,608 2,480 2,574 2,699 2,638 2,530 2,647 2,779 2,685 2,733 2,703 2598.36 2,661 -1.1% -3.9% 2.4%
      Previous 2,546 2,666 2,757 2,535 2,492 2,574 2,723 2,585 2,550 2,657 2,744 2,556 2,733 2,626 2,594 2,627 -3.9% -1.2% 1.3%
         Percent Change 3.0% 2.9% 2.8% 2.9% -0.5% 0.0% -0.9% 2.1% -0.8% -0.4% 1.3% 5.0% 0.0% 2.9% 0.2% 1.3%
   U.S. Total

U.S. Distillate 
      Current 4.26 3.90 3.96 3.86 3.90 3.80 3.79 4.02 4.01 3.88 3.85 4.02 4.04 4.00 3.88 3.94 -1.0% -3.0% 1.7%
      Previous 4.26 3.90 3.96 3.86 3.90 3.80 3.71 3.98 4.03 3.88 3.80 3.98 4.04 4.00 3.85 3.92 -1.0% -3.7% 2.0%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 2.1% 1.0% -0.5% -0.1% 1.3% 1.2% 0.0% 0.0% 0.8% 0.5%
   U.S. Jet Fuel 
      Current 1.46 1.57 1.60 1.57 1.50 1.61 1.68 1.65 1.52 1.58 1.64 1.64 1.47 1.55 1.61 1.60 5.3% 4.1% -0.9%

U.S. Federal GOM Marketed Production
      Current 3.27 3.54 3.81 3.49 3.48 3.34 3.23 3.31 3.35 3.33 3.21 3.2

In [43]:
prompt = f"""

Answer the question using only the information provided below.

Context:

{retrieved_context}

Question:

{user_question}

Answer:

"""

In [44]:
response = generator(
    prompt,
    max_length=200,
    do_sample=False
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [45]:
answer = response[0]["generated_text"]

print(answer)



Answer the question using only the information provided below.

Context:

U.S. Industrial
      Current 2,623 2,743 2,835 2,608 2,480 2,574 2,699 2,638 2,530 2,647 2,779 2,685 2,733 2,703 2598.36 2,661 -1.1% -3.9% 2.4%
      Previous 2,546 2,666 2,757 2,535 2,492 2,574 2,723 2,585 2,550 2,657 2,744 2,556 2,733 2,626 2,594 2,627 -3.9% -1.2% 1.3%
         Percent Change 3.0% 2.9% 2.8% 2.9% -0.5% 0.0% -0.9% 2.1% -0.8% -0.4% 1.3% 5.0% 0.0% 2.9% 0.2% 1.3%
   U.S. Total

U.S. Distillate 
      Current 4.26 3.90 3.96 3.86 3.90 3.80 3.79 4.02 4.01 3.88 3.85 4.02 4.04 4.00 3.88 3.94 -1.0% -3.0% 1.7%
      Previous 4.26 3.90 3.96 3.86 3.90 3.80 3.71 3.98 4.03 3.88 3.80 3.98 4.04 4.00 3.85 3.92 -1.0% -3.7% 2.0%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 2.1% 1.0% -0.5% -0.1% 1.3% 1.2% 0.0% 0.0% 0.8% 0.5%
   U.S. Jet Fuel 
      Current 1.46 1.57 1.60 1.57 1.50 1.61 1.68 1.65 1.52 1.58 1.64 1.64 1.47 1.55 1.61 1.60 5.3% 4.1% -0.9%

U.S. Federal GOM Marketed Production
      Current 3.

In [46]:
print("Question:")

print(user_question)

Question:
2


In [47]:
print("Generated Answer:")

print(answer)

Generated Answer:


Answer the question using only the information provided below.

Context:

U.S. Industrial
      Current 2,623 2,743 2,835 2,608 2,480 2,574 2,699 2,638 2,530 2,647 2,779 2,685 2,733 2,703 2598.36 2,661 -1.1% -3.9% 2.4%
      Previous 2,546 2,666 2,757 2,535 2,492 2,574 2,723 2,585 2,550 2,657 2,744 2,556 2,733 2,626 2,594 2,627 -3.9% -1.2% 1.3%
         Percent Change 3.0% 2.9% 2.8% 2.9% -0.5% 0.0% -0.9% 2.1% -0.8% -0.4% 1.3% 5.0% 0.0% 2.9% 0.2% 1.3%
   U.S. Total

U.S. Distillate 
      Current 4.26 3.90 3.96 3.86 3.90 3.80 3.79 4.02 4.01 3.88 3.85 4.02 4.04 4.00 3.88 3.94 -1.0% -3.0% 1.7%
      Previous 4.26 3.90 3.96 3.86 3.90 3.80 3.71 3.98 4.03 3.88 3.80 3.98 4.04 4.00 3.85 3.92 -1.0% -3.7% 2.0%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 2.1% 1.0% -0.5% -0.1% 1.3% 1.2% 0.0% 0.0% 0.8% 0.5%
   U.S. Jet Fuel 
      Current 1.46 1.57 1.60 1.57 1.50 1.61 1.68 1.65 1.52 1.58 1.64 1.64 1.47 1.55 1.61 1.60 5.3% 4.1% -0.9%

U.S. Federal GOM Marketed Productio

In [48]:
another_question = input("Ask another question: ")

Ask another question: 5


In [49]:
another_embedding = embedding_model.encode(
    [another_question],
    convert_to_numpy=True
)

distances, indices = index.search(
    another_embedding,
    3
)

In [50]:
context = ""

for idx in indices[0]:

    context += chunks[idx]

    context += "\n\n"

prompt = f"""

Context:

{context}

Question:

{another_question}

Answer:

"""

response = generator(
    prompt,
    max_length=200,
    do_sample=False
)

print(response[0]["generated_text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Context:

Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -1.7% -3.0% -0.2% 0.3% -0.5% -0.3% 0.0% 0.0% -1.0% -0.2%
   U.S. Commercial 
      Current 15.93 5.80 4.42 9.02 13.43 5.99 4.59 9.96 14.60 6.10 4.56 10.48 9.50 8.76 8.48 8.91 -7.7% -3.2% 5.1%
      Previous 15.93 5.80 4.42 9.02 13.43 5.98 4.60 9.99 14.61 6.13 4.54 10.20 9.50 8.76 8.49 8.85 -7.7% -3.1% 4.1%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.4% -0.3% -0.1% -0.5% 0.4% 2.8% 0.0% 0.0% -0.1% 0.7%
   U.S. Industrial

Previous 65,228 66,799 69,005 72,488 73,305 74,185 74,958 80,405 81,051 81,853 82,716 89,497 64,694 72,488 80,405 89,497 12.0% 10.9% 11.3%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.3% -1.0% -1.0% -1.0% -1.2% -0.9% 0.0% 0.0% -1.0% -0.9%
   Wood Biomass
      Current 3,278 3,278 3,235 3,185 3,185 3,187 3,169 3,177 3,177 3,177 3,219 3,219 3,278 3,185 3,177 3,219 -2.8% -0.3% 1.3%

U.S. Distillate 
      Current 4.26 3.90 3.96 3.86 3.90 3.80 3.79 4.02 4.01 3.88 3.85 4.02 4.04 4.00 3.88 3.94 

In [51]:
example_questions = [

    "What is the main objective of the document?",

    "Give a short summary of the document.",

    "What are the important topics discussed?"

]

In [52]:
for question in example_questions:

    print("="*80)

    print("Question:")

    print(question)

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        question_embedding,
        3
    )

    context = ""

    for idx in indices[0]:

        context += chunks[idx]

        context += "\n\n"

    prompt = f"""

Context:

{context}

Question:

{question}

Answer:

"""

    response = generator(
        prompt,
        max_length=200,
        do_sample=False
    )

    print("\nAnswer:\n")

    print(response[0]["generated_text"])

    print("\n")

Question:
What is the main objective of the document?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:



Context:

OECD (Organization for Economic Cooperation and Development) Consumption
      Current 46.63 45.64 46.92 46.46 46.72 45.97 46.64 47.13 47.02 45.95 46.90 47.58 45.86 46.41 46.62 46.87 1.2% 0.4% 0.5%
      Previous 46.63 45.64 46.92 46.46 46.72 45.98 46.52 47.15 46.97 45.91 46.84 47.44 45.86 46.41 46.59 46.79 1.2% 0.4% 0.4%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.3% 0.0% 0.1% 0.1% 0.1% 0.3% 0.0% 0.0% 0.1% 0.2%
   Non-OECD Consumption

Previous 94.73 95.48 96.46 96.50 95.53 95.51 96.26 97.31 96.49 97.42 97.70 98.10 93.35 95.80 96.16 97.43 2.6% 0.4% 1.3%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.1% -0.1% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0%
World Crude Oil and Liquid Fuels Consumption (million barrels per day)
   OECD (Organization for Economic Cooperation and Development) Consumption

Previous 26.76 26.51 26.89 27.12 26.97 25.91 26.21 26.48 26.47 26.52 26.52 27.07 25.81 26.82 26.39 26.65 3.9% -1.6% 1.0%
         Percent Change 0.0% 0

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:



Context:

OECD (Organization for Economic Cooperation and Development) Consumption
      Current 46.63 45.64 46.92 46.46 46.72 45.97 46.64 47.13 47.02 45.95 46.90 47.58 45.86 46.41 46.62 46.87 1.2% 0.4% 0.5%
      Previous 46.63 45.64 46.92 46.46 46.72 45.98 46.52 47.15 46.97 45.91 46.84 47.44 45.86 46.41 46.59 46.79 1.2% 0.4% 0.4%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.3% 0.0% 0.1% 0.1% 0.1% 0.3% 0.0% 0.0% 0.1% 0.2%
   Non-OECD Consumption

Previous 94.73 95.48 96.46 96.50 95.53 95.51 96.26 97.31 96.49 97.42 97.70 98.10 93.35 95.80 96.16 97.43 2.6% 0.4% 1.3%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.1% -0.1% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0%
World Crude Oil and Liquid Fuels Consumption (million barrels per day)
   OECD (Organization for Economic Cooperation and Development) Consumption

U.S. Federal GOM Marketed Production
      Current 3.27 3.54 3.81 3.49 3.48 3.34 3.23 3.31 3.35 3.33 3.21 3.22 3.43 3.53 3.34 3.28 2.8% -5.4% -1.8%
   

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:



Context:

OECD (Organization for Economic Cooperation and Development) Consumption
      Current 46.63 45.64 46.92 46.46 46.72 45.97 46.64 47.13 47.02 45.95 46.90 47.58 45.86 46.41 46.62 46.87 1.2% 0.4% 0.5%
      Previous 46.63 45.64 46.92 46.46 46.72 45.98 46.52 47.15 46.97 45.91 46.84 47.44 45.86 46.41 46.59 46.79 1.2% 0.4% 0.4%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.3% 0.0% 0.1% 0.1% 0.1% 0.3% 0.0% 0.0% 0.1% 0.2%
   Non-OECD Consumption

Previous 94.73 95.48 96.46 96.50 95.53 95.51 96.26 97.31 96.49 97.42 97.70 98.10 93.35 95.80 96.16 97.43 2.6% 0.4% 1.3%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.1% -0.1% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0%
World Crude Oil and Liquid Fuels Consumption (million barrels per day)
   OECD (Organization for Economic Cooperation and Development) Consumption

Previous 1423.9 1234.5 1338.0 1251.1 1321.3 1197.6 1348.6 1302.7 1365.5 1216.9 1314.0 1310.1 5,394 5,248 5,170 5,207 -2.7% -1.5% 0.7%
         Percent 

In [53]:
print("Retrieved Context Example:\n")

print(context)

Retrieved Context Example:

OECD (Organization for Economic Cooperation and Development) Consumption
      Current 46.63 45.64 46.92 46.46 46.72 45.97 46.64 47.13 47.02 45.95 46.90 47.58 45.86 46.41 46.62 46.87 1.2% 0.4% 0.5%
      Previous 46.63 45.64 46.92 46.46 46.72 45.98 46.52 47.15 46.97 45.91 46.84 47.44 45.86 46.41 46.59 46.79 1.2% 0.4% 0.4%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.3% 0.0% 0.1% 0.1% 0.1% 0.3% 0.0% 0.0% 0.1% 0.2%
   Non-OECD Consumption

Previous 94.73 95.48 96.46 96.50 95.53 95.51 96.26 97.31 96.49 97.42 97.70 98.10 93.35 95.80 96.16 97.43 2.6% 0.4% 1.3%
         Percent Change 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% -0.1% -0.1% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0% 0.0%
World Crude Oil and Liquid Fuels Consumption (million barrels per day)
   OECD (Organization for Economic Cooperation and Development) Consumption

Previous 1423.9 1234.5 1338.0 1251.1 1321.3 1197.6 1348.6 1302.7 1365.5 1216.9 1314.0 1310.1 5,394 5,248 5,170 5,207 -2.7% -1.5% 0.7%
         Pe

In [55]:
print("Total Document Chunks :", len(chunks))

print("Embedding Dimension :", embeddings.shape[1])

print("Total Stored Vectors :", index.ntotal)

Total Document Chunks : 89
Embedding Dimension : 384
Total Stored Vectors : 89


In [54]:
faiss.write_index(
    index,
    "rag_vector_database.index"
)

In [56]:
np.save(
    "document_embeddings.npy",
    embeddings
)

In [57]:
print("Vector Database Saved Successfully")

Vector Database Saved Successfully


In [58]:
print("Embeddings Saved Successfully")

Embeddings Saved Successfully


In [59]:
from google.colab import files

files.download("rag_vector_database.index")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
files.download("document_embeddings.npy")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [61]:
print("="*80)

print("Document Question Answering System")

print("="*80)

print("PDF Loaded Successfully")

print("Document Chunking Completed")

print("Embeddings Generated")

print("FAISS Vector Database Created")

print("Retrieval Completed")

print("Answer Generation Completed")

print("="*80)

Document Question Answering System
PDF Loaded Successfully
Document Chunking Completed
Embeddings Generated
FAISS Vector Database Created
Retrieval Completed
Answer Generation Completed


# Conclusion

In this project, I developed a simple Retrieval-Augmented Generation (RAG) system that can answer questions from custom PDF documents. Instead of relying only on the language model's existing knowledge, the system first retrieves the most relevant information from the uploaded document and then generates an answer based on that context.

The project involved reading a PDF, splitting it into smaller chunks, converting the text into embeddings, and storing those embeddings in a FAISS vector database. Whenever a user asks a question, the system searches for the most relevant chunks and provides them to the language model to generate an accurate response.

This project helped me understand how retrieval and generation work together in modern AI systems. It also provided practical experience with document processing, vector databases, semantic search, and large language models. RAG is widely used in AI-powered chatbots, enterprise knowledge assistants, search systems, and document-based question answering applications.